# 03 Validation

In [1]:
from google.colab import drive
drive.mount('/content/drive')

import os
os.chdir("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation/")

Mounted at /content/drive


In [2]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/Gates Foundation/Building Dataset Validation")
CONFIG_PATH  = PROJECT_ROOT / "configs/validation_configs.yaml"
TRACKER_PATH = PROJECT_ROOT / "data/02_interim/aoi_tracker.csv"

# Set to a list of dataset_folder_name values to limit which AOIs run,
# or leave as None to run everything
AOI_FILTER = None
# AOI_FILTER = ["ant-curacao", "ssd-juba"]

SKIP_EXISTING = False   # True = skip AOIs whose output parquets already exist
DRY_RUN       = False   # True = print plan only, do no work

In [3]:
import sys, os
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

In [4]:
import logging
from src.validator import load_config, load_tracker, process_aoi, already_done

for handler in logging.root.handlers[:]:
    logging.root.removeHandler(handler)
    
logging.basicConfig(
    level=logging.INFO,
    format="%(asctime)s  %(levelname)-8s  %(message)s",
    datefmt="%H:%M:%S",
    force=True,
)


cfg        = load_config(CONFIG_PATH)
root       = PROJECT_ROOT
tracker_df = load_tracker(TRACKER_PATH, aoi_filter=AOI_FILTER)

print(f"AOIs queued: {len(tracker_df)}")
display(tracker_df[["dataset_folder_name", "aoi_file_name", "reference_file_name"]])

16:11:31  INFO      Tracker: 56 AOI rows selected for processing.


AOIs queued: 56


,dataset_folder_name,aoi_file_name,reference_file_name
0,ant-curacao,curacao_aoi.geojson,curacao_hotosm.geojson
1,blz-burrell-boom,burrell-boom_aoi.geojson,burrell-boom_hotosm.geojson
2,bra-nova-sussuarana,nova-sussuarana_aoi.geojson,nova-sussuarana_hotosm.geojson
3,col-san-antonio-de-prado,san-antonio-de-prado_aoi.geojson,san-antonio-de-prado_hotosm.geojson
4,cvg-saint-vincent-grenadines,saint_vincent_grenadines_aoi.geojson,saint_vincent_grenadines_ref.geojson
5,dom-dominica,dominica_aoi.geojson,NaN
6,gha-aiyim-sraha,aiyim-sraha_aoi.geojson,aiyim-sraha_hotosm.geojson
7,gha-dansoman,dansoman_aoi.geojson,dansoman_hotosm.geojson
8,gha-nawuni,nawuni_aoi.geojson,nawuni_hotosm.geojson
9,gha-sawla-tuna,sawla-tuna_aoi.geojson,sawla-tuna_hotosm.geojson


In [ ]:
if DRY_RUN:
    print(f"\nDry-run — {len(tracker_df)} AOIs:\n")
    for _, row in tracker_df.iterrows():
        slug = row["dataset_folder_name"].lower()
        md   = root / "outputs" / "metrics" / slug
        done = already_done(md)
        tag  = "SKIP" if (SKIP_EXISTING and done) else ("DONE" if done else "PENDING")
        print(f"  {tag:7s}  {row['dataset_folder_name']}")
else:
    print("DRY_RUN=False — proceeding to execution.")

DRY_RUN=False — proceeding to execution.


: 

In [6]:
import traceback
import pandas as pd

if not DRY_RUN:
    results = {}

    for _, row in tracker_df.iterrows():
        folder    = row["dataset_folder_name"]
        city_slug = folder.lower()
        md        = root / "outputs" / "metrics" / city_slug

        print(f"\nProcessing {folder}...")
        if SKIP_EXISTING and already_done(md):
            print(f"[{folder}] skipped (already done)")
            results[folder] = "skipped"
            continue

        try:
            ok = process_aoi(row, cfg, root)
            results[folder] = "ok" if ok else "no_data"
        except Exception:
            print(f"[{folder}] ERROR")
            traceback.print_exc()
            results[folder] = "error"

    # Summary
    summary = pd.DataFrame(
        [(k, v) for k, v in results.items()],
        columns=["aoi", "status"]
    )
    print(f"\nDone — {len(summary)} AOIs processed.\n")
    display(summary.groupby("status")["aoi"].count().rename("count").to_frame())
    display(summary)
 


Processing ant-curacao...


16:11:33  INFO      ━━━━  AOI: ant-curacao  ━━━━
16:11:34  INFO      [ant-curacao] 1 tiles generated.
16:11:34  INFO      Created 1 records
16:11:37  INFO      [ant-curacao] Reference buildings: 15054
16:11:37  INFO      [ant-curacao / overture] Loading candidate: ant_curacao_overture_building.parquet
16:11:39  INFO      [ant-curacao / overture] Candidate buildings: 19169


: 

: 